# Plot sfc_mean2t (HEALPix) — January 1990 (Robinson)

This notebook reproduces the Robinson-projection plot (Cartopy) for the HEALPix **NESTED** field `mean2t` from:
- `/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t/sfc_mean2t_1990.nc`

## Important: environment on Levante
To run this notebook on Levante, use a kernel/environment that has `cartopy` (and friends).
We used the module: `esmvaltool/2.11.0` (Python 3.10).


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from astropy_healpix import HEALPix
import astropy.units as u


In [ ]:
# Input/output paths
path_nc = '/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t/sfc_mean2t_1990.nc'
out_png = '/work/ab0995/a270135/MN5/projt319/a3be_netcdf/sfc_mean2t_figures/sfc_mean2t_1990_Jan_robinson_250_315.png'


In [ ]:
# Load January (time index 0)
ds = xr.open_dataset(path_nc)
vals_hp = ds['mean2t'].isel(time=0, height=0).values.astype('float64')
npix = vals_hp.size
nside = int(round(np.sqrt(npix / 12.0)))
assert 12 * nside * nside == npix, f'npix={npix} is not 12*nside^2'
print('npix:', npix, 'nside:', nside)


In [ ]:
# Regrid HEALPix (NESTED) -> regular 1° lat/lon grid (nearest-neighbor sampling)
lon = np.arange(-180.0, 180.0, 1.0)
lat = np.arange(-90.0, 90.0 + 1.0, 1.0)
lon2d, lat2d = np.meshgrid(lon, lat)

hp = HEALPix(nside=nside, order='nested', frame=None)
idx = hp.lonlat_to_healpix(lon2d * u.deg, lat2d * u.deg)
grid_vals = vals_hp[idx]
print('grid shape:', grid_vals.shape)


In [ ]:
# Plot (Robinson projection)
fig = plt.figure(figsize=(13, 6))
ax = plt.axes(projection=ccrs.Robinson())

im = ax.pcolormesh(
    lon2d, lat2d, grid_vals,
    transform=ccrs.PlateCarree(),
    cmap='RdBu_r',
    vmin=250, vmax=315,
    shading='auto'
)

ax.coastlines(linewidth=0.7)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, color='gray', alpha=0.2)
ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.5)

cb = plt.colorbar(im, orientation='horizontal', pad=0.05, fraction=0.05)
cb.set_label('mean2t (K)', fontsize=10)

plt.title('mean2t (2m air temperature) — January 1990 (HEALPix NESTED → 1° grid)', fontsize=12)
plt.tight_layout()
plt.savefig(out_png, dpi=200)
plt.show()
print('saved:', out_png)


## Global mean 2m temperature (annual mean per year) + trend

Compute one global-mean value per available yearly file in `sfc_mean2t/`, then plot a time series and linear trend.


In [ ]:
import glob
import os
import re

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
